# Task 1 — Baseline Pix2Pix on MoNuSeg

**Computer Vision II — Master in Artificial Intelligence (2025/2026)**

This notebook implements a **baseline Pix2Pix** image-to-image translation model
following Isola *et al.* (CVPR, 2017) on the **MoNuSeg** dataset (500 paired
samples, each containing a histopathology image and the corresponding nuclei
binary mask).

**Translation direction:** `label map (binary nuclei mask)  →  realistic histology image`.

The notebook is intentionally light: the heavy lifting (model definitions,
dataset, training loop, evaluation) lives in the shared `models/` and
`util/` packages, the same layout used by `task2_improvements.ipynb`.

Pipeline:
1. Load and process the MoNuSeg dataset (split each PNG into the two halves).
2. Split into train / validation / test (70 / 15 / 15).
3. Train the baseline U-Net + 70×70 PatchGAN with the Pix2Pix objective.
4. Evaluate on the test split (L1, PSNR, SSIM, FID).
5. Re-train **with paired data augmentation** ("augmented generation") and
   compare both runs.

## 1. Imports and configuration

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import random
from pathlib import Path

import numpy as np
import torch

from models.pix2pix_model import build_baseline_pix2pix
from util.data import make_loaders, denorm
from util.training import train_pix2pix, plot_losses
from util.metrics import evaluate_test_set, show_qualitative


def resolve_device():
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = resolve_device()
print("Device:", DEVICE)

# ---------- Hyperparameters ----------
DATA_DIR     = Path("MoNuSeg")
IMG_SIZE     = 256
BATCH_SIZE   = 16
NUM_WORKERS  = 0
EPOCHS       = 200
LR           = 2e-4
BETAS        = (0.5, 0.999)
LAMBDA_L1    = 100.0
NGF          = 64
NDF          = 64

SAVE_DIR_BASE = Path("checkpoints_baseline");      SAVE_DIR_BASE.mkdir(exist_ok=True)
SAMPLES_DIR_BASE = Path("samples_baseline");        SAMPLES_DIR_BASE.mkdir(exist_ok=True)
SAVE_DIR_AUG  = Path("checkpoints_baseline_aug");  SAVE_DIR_AUG.mkdir(exist_ok=True)
SAMPLES_DIR_AUG = Path("samples_baseline_aug");     SAMPLES_DIR_AUG.mkdir(exist_ok=True)

## 2. Dataset

Each PNG file in `MoNuSeg/` is a horizontally concatenated pair: the **left
half** is the real histopathology image (RGB) and the **right half** is the
corresponding **binary nuclei mask**. The dataset class lives in
[`util/data.py`](util/data.py); it splits each PNG, resizes both halves to
256×256 and normalizes to `[-1, 1]` so the generator can use a `tanh`
output, as in the original Pix2Pix.

In [ ]:
loaders = make_loaders(
    DATA_DIR,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    augment_train=False,
)
train_set    = loaders["train_set"]
val_set      = loaders["val_set"]
test_set     = loaders["test_set"]
train_loader = loaders["train_loader"]
val_loader   = loaders["val_loader"]
test_loader  = loaders["test_loader"]

print(f"Dataset size: {len(train_set) + len(val_set) + len(test_set)}")
print(f"  train: {len(train_set)}  | val: {len(val_set)}  | test: {len(test_set)}")
print(f"  batches/epoch (train): {len(train_loader)}")

### 2.1 Visual sanity check

A few random pairs from the training split to confirm that the left half is the
real histology image and the right half is the nuclei mask.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for col in range(4):
    label, real = train_set[col]
    axes[0, col].imshow(denorm(label).permute(1, 2, 0).numpy())
    axes[0, col].set_title("label (input)")
    axes[0, col].axis("off")
    axes[1, col].imshow(denorm(real).permute(1, 2, 0).numpy())
    axes[1, col].set_title("real (target)")
    axes[1, col].axis("off")
plt.tight_layout()
plt.show()

## 3. Models

The U-Net generator and 70×70 PatchGAN discriminator are defined in
[`models/pix2pix_model.py`](models/pix2pix_model.py). The factory
`build_baseline_pix2pix` instantiates both networks, applies the Pix2Pix
weight init (`N(0, 0.02)`) and moves them to the requested device.

In [ ]:
G, D = build_baseline_pix2pix(in_channels=3, out_channels=3, ngf=NGF, ndf=NDF, device=DEVICE)

with torch.no_grad():
    dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    out_g = G(dummy)
    out_d = D(dummy, out_g)
print("Generator     output:", tuple(out_g.shape))
print("Discriminator output:", tuple(out_d.shape))
n_params_g = sum(p.numel() for p in G.parameters())
n_params_d = sum(p.numel() for p in D.parameters())
print(f"#params  G: {n_params_g/1e6:.2f} M   D: {n_params_d/1e6:.2f} M")

## 4. Training — baseline (no data augmentation)

We train the U-Net + PatchGAN with the standard Pix2Pix objective: conditional
BCE adversarial loss + λ·L1 reconstruction. The training loop lives in
[`util/training.py`](util/training.py); per epoch we monitor validation L1
and persist the best checkpoint.

In [ ]:
history_base, best_val_base, best_epoch_base = train_pix2pix(
    G, D,
    train_loader=train_loader,
    val_loader=val_loader,
    val_set=val_set,
    device=DEVICE,
    epochs=EPOCHS,
    lr=LR,
    betas=BETAS,
    lambda_l1=LAMBDA_L1,
    save_dir=SAVE_DIR_BASE,
    samples_dir=SAMPLES_DIR_BASE,
)

### 4.1 Loss curves

In [ ]:
plot_losses(history_base)

## 5. Evaluation on the test set

We reload the **best** checkpoint (lowest validation L1) and report on the
test split using L1, PSNR, SSIM and FID. The evaluation utilities live in
[`util/metrics.py`](util/metrics.py).

In [ ]:
ckpt_base = torch.load(SAVE_DIR_BASE / "best.pt", map_location=DEVICE)
G.load_state_dict(ckpt_base["G"])
G.eval()
print(f"Loaded best generator (epoch {ckpt_base['epoch']}, val L1 = {ckpt_base['val_l1']:.4f})")

results_base = evaluate_test_set(G, test_loader, DEVICE)
print("\nTest-set metrics (baseline Pix2Pix on MoNuSeg)")
print("-" * 48)
for k, v in results_base.items():
    print(f"  {k:<5}: {v:.4f}")

### 5.1 Qualitative results

Side-by-side `label  |  generated  |  real` on a handful of test images.

In [ ]:
show_qualitative(G, test_set, DEVICE, n_show=6,
                 save_path=SAMPLES_DIR_BASE / "test_qualitative.png")

### 5.2 Persist the metrics

Save the test-set metrics as a small JSON next to the checkpoints — this file
is the artefact that Task 2 will compare against.

In [ ]:
import json

summary_base = {
    "dataset": "MoNuSeg",
    "model":   "Pix2Pix baseline (U-Net + 70x70 PatchGAN)",
    "augmentation": False,
    "epochs":  EPOCHS,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "lambda_l1": LAMBDA_L1,
    "best_epoch": int(ckpt_base["epoch"]),
    "test_metrics": {k: float(v) for k, v in results_base.items()},
}
with open(SAVE_DIR_BASE / "test_metrics.json", "w") as f:
    json.dump(summary_base, f, indent=2)
print("Saved", SAVE_DIR_BASE / "test_metrics.json")
print(json.dumps(summary_base, indent=2))

## 6. Augmented generation — paired data augmentation

As an improvement over the baseline, we re-train the **same** Pix2Pix
architecture with paired data augmentation on the training split. The
augmentation pipeline (in
[`util/augmentation.py`](util/augmentation.py), function
`augment_training_pair`) is the same routine used in
`task2_improvements.ipynb` and applies a single random affine transform
**synchronously** to (label, real) with probability 0.5:

* random rotation in `±15°`,
* random zoom-in in `[1.00, 1.15]`,
* random translation up to `±5%` along each axis.

Nearest-neighbour interpolation is used for the label (so binary
boundaries stay crisp) and bilinear for the real image. Background fill
is `-1.0` to match the `[-1, 1]` normalization. The augmentation
samples from a continuous family of transforms, so each epoch sees a
different geometric configuration of every training pair.

The val/test splits use the **same clean transform** as before, so the
metrics remain directly comparable to section 5.

In [ ]:
loaders_aug = make_loaders(
    DATA_DIR,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    augment_train=True,
)
train_set_aug    = loaders_aug["train_set"]
train_loader_aug = loaders_aug["train_loader"]
# val/test loaders are clean by construction, but we reuse the same split
val_loader_aug   = loaders_aug["val_loader"]
test_loader_aug  = loaders_aug["test_loader"]
val_set_aug      = loaders_aug["val_set"]
test_set_aug     = loaders_aug["test_set"]

print(f"Train (augmented): {len(train_set_aug)} pairs, augment_train=True")

### 6.1 Visualize augmented training samples

The same training index now produces a different (label, real) pair on each
`__getitem__` call thanks to the random transforms. Two consecutive draws of
the first index:

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for row in range(2):
    for col in range(4):
        label, real = train_set_aug[col]
        axes[row, col].imshow(denorm(real).permute(1, 2, 0).numpy())
        axes[row, col].set_title(f"draw {row+1} — idx {col}")
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()

### 6.2 Train the augmented model

Same hyper-parameters and budget as section 4 — the only thing that changes is
the training data pipeline.

In [ ]:
G_aug, D_aug = build_baseline_pix2pix(in_channels=3, out_channels=3, ngf=NGF, ndf=NDF, device=DEVICE)

history_aug, best_val_aug, best_epoch_aug = train_pix2pix(
    G_aug, D_aug,
    train_loader=train_loader_aug,
    val_loader=val_loader_aug,
    val_set=val_set_aug,
    device=DEVICE,
    epochs=EPOCHS,
    lr=LR,
    betas=BETAS,
    lambda_l1=LAMBDA_L1,
    save_dir=SAVE_DIR_AUG,
    samples_dir=SAMPLES_DIR_AUG,
)

### 6.3 Loss curves (augmented)

In [ ]:
plot_losses(history_aug)

### 6.4 Evaluation on the test set (augmented)

In [ ]:
ckpt_aug = torch.load(SAVE_DIR_AUG / "best.pt", map_location=DEVICE)
G_aug.load_state_dict(ckpt_aug["G"])
G_aug.eval()
print(f"Loaded best augmented generator (epoch {ckpt_aug['epoch']}, val L1 = {ckpt_aug['val_l1']:.4f})")

results_aug = evaluate_test_set(G_aug, test_loader_aug, DEVICE)
print("\nTest-set metrics (Pix2Pix + augmentation on MoNuSeg)")
print("-" * 48)
for k, v in results_aug.items():
    print(f"  {k:<5}: {v:.4f}")

### 6.5 Qualitative results (augmented)

In [ ]:
show_qualitative(G_aug, test_set_aug, DEVICE, n_show=6,
                 save_path=SAMPLES_DIR_AUG / "test_qualitative.png")

### 6.6 Side-by-side comparison and persisted metrics

A direct comparison of both runs on the same test split.

In [ ]:
import json

summary_aug = {
    "dataset": "MoNuSeg",
    "model":   "Pix2Pix baseline (U-Net + 70x70 PatchGAN)",
    "augmentation": True,
    "augment_ops": ["rotation(+-15deg)", "zoom_in([1.00,1.15])", "translate(+-5%)"],
    "epochs":  EPOCHS,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "lambda_l1": LAMBDA_L1,
    "best_epoch": int(ckpt_aug["epoch"]),
    "test_metrics": {k: float(v) for k, v in results_aug.items()},
}
with open(SAVE_DIR_AUG / "test_metrics.json", "w") as f:
    json.dump(summary_aug, f, indent=2)

print("Saved", SAVE_DIR_AUG / "test_metrics.json")

# --- Comparison table ---
print()
print(f"{'Metric':<8} {'Baseline':>12} {'+ Augment':>12} {'Δ (better)':>14}")
print("-" * 50)
better_dir = {"L1": "lower", "FID": "lower", "PSNR": "higher", "SSIM": "higher"}
for k in ["L1", "PSNR", "SSIM", "FID"]:
    a, b = results_base[k], results_aug[k]
    delta = b - a
    if better_dir[k] == "lower":
        sign = "↓" if delta < 0 else "↑"
    else:
        sign = "↑" if delta > 0 else "↓"
    print(f"{k:<8} {a:>12.4f} {b:>12.4f} {sign} {abs(delta):>10.4f}")

## Summary

The baseline Pix2Pix run gives us the reference metrics that **task 2**
compares against. The augmented run uses the same model + hyper-parameters
and only swaps the training data pipeline — any change in the test-set
metrics is therefore attributable to the augmentation strategy alone.

Caveats:

* The MoNuSeg train set is small (350 pairs). The shared affine
  augmentation (rotation, zoom-in, translation) samples from a continuous
  family of transforms, so every epoch sees a different geometric
  configuration of each training pair. This usually helps the generator
  avoid memorising the training set — at the cost of a slower-converging
  validation L1.
* Pixel-wise metrics (L1, PSNR, SSIM) and perceptual metrics (FID) can move
  in opposite directions: augmentation tends to improve FID more than L1
  because it pushes the generator toward realistic textures rather than
  toward the exact pixel layout of a specific training sample.